In [ ]:
# libraries used
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.svm import OneClassSVM
from sklearn.metrics import root_mean_squared_error, r2_score

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

# imported games.csv which is the steam games dataset
gamesDf = pd.read_csv('games.csv')

# revenue for games with 0 games sold (estimated owners) will be 0 so removed them
gamesDf = gamesDf[gamesDf['Estimated owners'] != 0].reset_index(drop=True)

# revenue for games with a price of $0 will be 0 so removed them
gamesDf = gamesDf[gamesDf['Price'] != 0].reset_index(drop=True)

# target is revenue: (games sold * price) * 0.7 (steam takes 30%)
targetDf = pd.DataFrame()
targetDf['revenue'] = gamesDf['Estimated owners'] * gamesDf['Price'] * .7

# relevant features for revenue prediction
featuresDf = gamesDf[['Name', 'Release date', 'Estimated owners', 'Price', 'Score rank',
                       'Recommendations', 'Average playtime two weeks', 'Median playtime forever',
                       'Median playtime two weeks', 'Publishers', 'Categories', 'Genres', 'Tags']].copy()

featuresDf = featuresDf.fillna(0)

nonNumericalFeat = ['Publishers', 'Categories', 'Genres', 'Tags', 'Release date']
numericalFeat = ['Estimated owners', 'Price', 'Score rank', 'Recommendations',
                 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks']

# log transform skewed numerical features then min-max normalize
skewedFeat = ['Estimated owners', 'Score rank', 'Recommendations',
              'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks']
featuresDf[skewedFeat] = np.log1p(featuresDf[skewedFeat])
featuresDf[numericalFeat] = (featuresDf[numericalFeat] - featuresDf[numericalFeat].min()) / \
                             (featuresDf[numericalFeat].max() - featuresDf[numericalFeat].min())

# integer-encode categorical columns (replaces get_dummies — avoids 50k+ column explosion)
encoders = {}
vocab_sizes = {}
for col in nonNumericalFeat:
    le = LabelEncoder()
    featuresDf[col] = le.fit_transform(featuresDf[col].astype(str))
    encoders[col] = le
    vocab_sizes[col] = len(le.classes_)

# combine numerical + integer-encoded categoricals (small matrix now)
featurez = featuresDf[numericalFeat + nonNumericalFeat].copy()

# One-Class SVM outlier detection on numerical features only
x_num = featuresDf[numericalFeat].to_numpy()
svm = OneClassSVM(kernel='rbf', gamma=0.001, nu=0.03)
svm.fit(x_num)
labels = svm.predict(x_num)
inlier = labels == 1
outlier = labels == -1
numOutliers = (labels == -1).sum()
numInliers = (labels == 1).sum()

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 6))
ax0.scatter(x_num[:, 0], x_num[:, 1], s=5, alpha=0.3, color='blue', label='All points')
ax0.set_title("Original Data")
ax0.set_xlabel("Estimated owners")
ax0.set_ylabel("Price")
ax0.legend(markerscale=3)

ax1.scatter(x_num[inlier, 0], x_num[inlier, 1], s=5, alpha=0.3, color='blue', label='Inlier')
ax1.scatter(x_num[outlier, 0], x_num[outlier, 1], s=20, alpha=0.7, color='red', label='Outlier')
ax1.set_title("Detected Outliers using One Class SVM")
ax1.set_xlabel("Estimated owners")
ax1.set_ylabel("Price")
ax1.legend(markerscale=3)
plt.suptitle(f"One-Class SVM | nu={svm.nu}, gamma={svm.gamma} | {numOutliers} outliers ({100*numOutliers/len(labels):.1f}%)")
plt.tight_layout()
plt.show()
print("Outliers:", numOutliers, "out of", len(labels))

# remove outliers
featurez = featurez[inlier].reset_index(drop=True)
targetDf = targetDf[inlier].reset_index(drop=True)

# pairplot for numerical features
sns.pairplot(featurez[numericalFeat])
plt.show()

# train/validation/test split (70:15:15), log-transform revenue target
x = featurez
y = np.log1p(targetDf['revenue'])
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=66)
x_validation, x_test, y_validation, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=66)

# helper: split a DataFrame into [numerical_array, cat1_array, cat2_array, ...]
def prepare_inputs(df):
    return [df[numericalFeat].values.astype('float32')] + \
           [df[col].values.astype('int32') for col in nonNumericalFeat]

# --- Keras Functional Model with Embedding layers ---
EMBED_DIM = 16  # low-dimensional dense representation per category

num_input = Input(shape=(len(numericalFeat),), name='numerical')
num_x = Dense(64, activation='relu')(num_input)

cat_inputs = []
cat_outputs = []
for col in nonNumericalFeat:
    inp = Input(shape=(1,), name=col)
    emb = Embedding(input_dim=vocab_sizes[col] + 1, output_dim=EMBED_DIM, name=f'emb_{col}')(inp)
    flat = Flatten()(emb)
    cat_inputs.append(inp)
    cat_outputs.append(flat)

merged = Concatenate()([num_x] + cat_outputs)
z = Dense(256, activation='relu')(merged)
z = Dense(128, activation='relu')(z)
z = Dense(32, activation='relu')(z)
output = Dense(1, activation='linear')(z)

model = Model(inputs=[num_input] + cat_inputs, outputs=output)
model.compile(optimizer='adam', loss='mse')
model.summary()

# train
print("Training model...")
history = model.fit(
    prepare_inputs(x_train), y_train,
    validation_data=(prepare_inputs(x_validation), y_validation),
    epochs=100, batch_size=2000, verbose=1
)

# predictions — reverse log transform
y_pred_log = model.predict(prepare_inputs(x_test))
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log).flatten()

# RMSE and R²
rmse = root_mean_squared_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)
print(f"\nRMSE: ${rmse:,.2f}")
print(f"R²:   {r2:.4f}")

# training curve
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Training and Validation Loss (MSE)', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.grid(True)
plt.suptitle("Model Training Performance", fontsize=16)
plt.tight_layout()
plt.show()
